# Phase 8: Baseline Model Training (XGBoost)
# المرحلة 8: تدريب النموذج الأساسي (XGBoost)

This notebook trains a tuned **XGBoost baseline** on gene-level splits,
uses categorical encoding, and reports train/validation/test performance.

In [ ]:
import warnings
warnings.filterwarnings('ignore')

from pathlib import Path
import sys

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder
from sklearn.metrics import confusion_matrix

repo_root = Path.cwd().resolve()
if repo_root.name == 'notebooks':
    repo_root = repo_root.parent

src_path = repo_root / 'src'
if str(src_path) not in sys.path:
    sys.path.insert(0, str(src_path))

from models.xgboost_model import XGBTuningConfig, tune_xgboost
from evaluation import compute_classification_metrics, select_best_threshold

print(f'Repo root: {repo_root}')

## Step 1: Load Train / Validation / Test
## الخطوة 1: تحميل تقسيمات البيانات

In [ ]:
train_df = pd.read_parquet(repo_root / 'data/splits/train.parquet')
val_df = pd.read_parquet(repo_root / 'data/splits/val.parquet')
test_df = pd.read_parquet(repo_root / 'data/splits/test.parquet')

def split_summary(name, df):
    p = int((df['label'] == 1).sum())
    b = int((df['label'] == 0).sum())
    return {
        'Split': name,
        'Rows': len(df),
        'Genes': df['gene'].nunique(),
        'Pathogenic (1)': p,
        'Benign (0)': b,
        'Pathogenic %': (p / len(df)) * 100,
    }

pd.DataFrame([
    split_summary('Train', train_df),
    split_summary('Validation', val_df),
    split_summary('Test', test_df),
])

## Step 2: Feature Selection + Categorical Encoding
## الخطوة 2: اختيار السمات وترميز الكاتيجوريكل

- Numeric/bool features are kept
- Useful categorical columns are one-hot encoded (`chr`, `ref`, `alt`, `ref_aa`, ...)
- `gene` is **excluded intentionally** because split is gene-level (group key), and we avoid using gene identity directly as a shortcut feature
- Leakage-prone columns are excluded: `ClinicalSignificance`, `variant_key`

In [ ]:
excluded = {'label', 'variant_key', 'ClinicalSignificance', 'PhenotypeIDS', 'gene'}

numeric_cols = []
categorical_cols = []

for c in train_df.columns:
    if c in excluded:
        continue
    dtype = train_df[c].dtype
    if pd.api.types.is_numeric_dtype(dtype) or pd.api.types.is_bool_dtype(dtype):
        numeric_cols.append(c)
    elif (
        pd.api.types.is_object_dtype(dtype)
        or pd.api.types.is_string_dtype(dtype)
        or isinstance(dtype, pd.CategoricalDtype)
    ):
        categorical_cols.append(c)

print('Numeric columns:', len(numeric_cols))
print(numeric_cols)
print('\nCategorical columns:', len(categorical_cols))
print(categorical_cols)

In [ ]:
def make_ohe():
    try:
        return OneHotEncoder(handle_unknown='ignore', sparse_output=True, min_frequency=25)
    except TypeError:
        return OneHotEncoder(handle_unknown='ignore', sparse=True)

def prepare_with_encoding(train_df, val_df, test_df, numeric_cols, categorical_cols):
    selected = numeric_cols + categorical_cols

    X_train = train_df[selected].copy()
    X_val = val_df[selected].copy()
    X_test = test_df[selected].copy()

    # Numeric preprocessing
    if numeric_cols:
        for x in (X_train, X_val, X_test):
            x[numeric_cols] = x[numeric_cols].replace([np.inf, -np.inf], np.nan)

        medians = X_train[numeric_cols].median(numeric_only=True)
        X_train[numeric_cols] = X_train[numeric_cols].fillna(medians)
        X_val[numeric_cols] = X_val[numeric_cols].fillna(medians)
        X_test[numeric_cols] = X_test[numeric_cols].fillna(medians)

        bool_cols = [c for c in numeric_cols if pd.api.types.is_bool_dtype(X_train[c])]
        for c in bool_cols:
            X_train[c] = X_train[c].astype(np.int8)
            X_val[c] = X_val[c].astype(np.int8)
            X_test[c] = X_test[c].astype(np.int8)

    # Categorical preprocessing
    if categorical_cols:
        for x in (X_train, X_val, X_test):
            for c in categorical_cols:
                x[c] = x[c].fillna('MISSING').astype(str)

    transformers = []
    if numeric_cols:
        transformers.append(('num', 'passthrough', numeric_cols))
    if categorical_cols:
        transformers.append(('cat', make_ohe(), categorical_cols))

    preprocessor = ColumnTransformer(transformers=transformers, remainder='drop', sparse_threshold=1.0)

    X_train_enc = preprocessor.fit_transform(X_train)
    X_val_enc = preprocessor.transform(X_val)
    X_test_enc = preprocessor.transform(X_test)

    try:
        encoded_feature_names = preprocessor.get_feature_names_out().tolist()
    except Exception:
        encoded_feature_names = [f'f_{i}' for i in range(X_train_enc.shape[1])]

    return X_train_enc, X_val_enc, X_test_enc, encoded_feature_names

X_train, X_val, X_test, encoded_feature_names = prepare_with_encoding(
    train_df, val_df, test_df, numeric_cols, categorical_cols
)

y_train = train_df['label'].astype(int)
y_val = val_df['label'].astype(int)
y_test = test_df['label'].astype(int)

print('Encoded feature count:', len(encoded_feature_names))

## Step 3: Hyperparameter Tuning
## الخطوة 3: توليف الهايبر باراميترز

In [ ]:
pos_count = int((y_train == 1).sum())
neg_count = int((y_train == 0).sum())
scale_pos_weight = neg_count / max(pos_count, 1)

config = XGBTuningConfig(
    n_trials=14,
    seed=42,
    n_estimators=2500,
    early_stopping_rounds=80,
)

best_model, best_params, tuning_history = tune_xgboost(
    X_train, y_train, X_val, y_val,
    config=config,
    scale_pos_weight=float(scale_pos_weight),
)

print('Best hyperparameters:')
for k, v in best_params.items():
    print(f'  {k}: {v}')

tuning_history.head(10)

## Step 4: Threshold + Evaluation
## الخطوة 4: العتبة والتقييم

In [ ]:
val_prob = best_model.predict_proba(X_val)[:, 1]
best_threshold, threshold_curve = select_best_threshold(y_val, val_prob)

rows = []
for split_name, X, y in [
    ('Train', X_train, y_train),
    ('Validation', X_val, y_val),
    ('Test', X_test, y_test),
]:
    prob = best_model.predict_proba(X)[:, 1]
    m = compute_classification_metrics(y, prob, threshold=best_threshold)
    m['Split'] = split_name
    rows.append(m)

metrics_df = pd.DataFrame(rows)[['Split', 'threshold', 'roc_auc', 'pr_auc', 'accuracy',
                                 'balanced_accuracy', 'precision', 'recall', 'f1', 'mcc',
                                 'brier_loss', 'log_loss', 'tn', 'fp', 'fn', 'tp', 'support']]

print(f'Best validation threshold: {best_threshold:.3f}')
metrics_df.round(4)

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

axes[0].plot(threshold_curve['threshold'], threshold_curve['f1'], color='#2e86de', linewidth=2)
axes[0].axvline(best_threshold, color='red', linestyle='--', label=f'Best={best_threshold:.3f}')
axes[0].set_title('Validation F1 vs Threshold')
axes[0].set_xlabel('Threshold')
axes[0].set_ylabel('F1')
axes[0].legend()

metrics_df.set_index('Split')[['roc_auc', 'pr_auc']].plot(kind='bar', ax=axes[1], color=['#27ae60', '#8e44ad'])
axes[1].set_title('Ranking Metrics by Split')
axes[1].set_ylim(0.0, 1.0)
axes[1].set_ylabel('Score')
axes[1].tick_params(axis='x', rotation=0)

test_prob = best_model.predict_proba(X_test)[:, 1]
test_pred = (test_prob >= best_threshold).astype(int)
cm = confusion_matrix(y_test, test_pred, labels=[0, 1])
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=axes[2], cbar=False)
axes[2].set_title('Test Confusion Matrix')
axes[2].set_xlabel('Predicted')
axes[2].set_ylabel('True')
axes[2].set_xticklabels(['Benign (0)', 'Pathogenic (1)'])
axes[2].set_yticklabels(['Benign (0)', 'Pathogenic (1)'])

plt.tight_layout()
plt.show()

In [ ]:
checkpoint_dir = repo_root / 'results/checkpoints'
metrics_dir = repo_root / 'results/metrics'

checkpoint_dir.mkdir(parents=True, exist_ok=True)
metrics_dir.mkdir(parents=True, exist_ok=True)

model_path = checkpoint_dir / 'xgboost_best.ubj'
metrics_path = metrics_dir / 'xgboost_split_metrics.csv'
tuning_path = metrics_dir / 'xgboost_tuning_history.csv'
threshold_curve_path = metrics_dir / 'xgboost_val_threshold_curve.csv'
features_path = metrics_dir / 'xgboost_feature_columns.csv'
params_path = metrics_dir / 'xgboost_best_params.csv'

best_model.save_model(str(model_path))
metrics_df.to_csv(metrics_path, index=False)
tuning_history.to_csv(tuning_path, index=False)
threshold_curve.to_csv(threshold_curve_path, index=False)
pd.DataFrame({'encoded_feature': encoded_feature_names}).to_csv(features_path, index=False)
pd.DataFrame([{**best_params, 'best_threshold': best_threshold,
               'n_numeric_features': len(numeric_cols),
               'n_categorical_features': len(categorical_cols),
               'n_encoded_features': len(encoded_feature_names)}]).to_csv(params_path, index=False)

print('Saved artifacts:')
print('  model:   ', model_path)
print('  metrics: ', metrics_path)
print('  tuning:  ', tuning_path)
print('  curve:   ', threshold_curve_path)
print('  features:', features_path)
print('  params:  ', params_path)